# T-A2_Magnificent7 — 数据获取

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-A2：美股"七姐妹"财务与股价全景对比 |
| 小组   | 第七组 |
| 成员   | 莫才有（7125210211）、刘帆（225210189）、王天正（325210247）、许玲敏（425210271）、高世杰（625210132）、曾垂健（725210291）、廖晟可（825210181） |
| GitHub | https://github.com/ZDC3/ds2026-G07-Magnificent-7 |
| Pages  | https://zdc3.github.io/ds2026-G07-Magnificent-7 |
| 日期   | 2026-05-16 |

## 任务说明
使用 yfinance 批量下载七家公司近 5 年日度复权收盘价，并获取财务摘要指标（PE、PB、市值、营收增速等）。保存到 data_raw 目录，并完成数据质量检查（缺失率、时间覆盖、重复记录、非正价格与最新日期一致性）。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import yfinance as yf

# 路径设置
project_dir = Path('.').resolve()
data_raw_dir = project_dir / 'data_raw'
data_raw_dir.mkdir(parents=True, exist_ok=True)

tickers = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'TSLA']

# 1) 批量下载股价（近 5 年，日度，复权）
prices = yf.download(
    tickers,
    period='5y',
    interval='1d',
    auto_adjust=True,
    progress=False
)['Close']

if prices.empty:
    raise ValueError('股价数据为空，请检查网络或 ticker 列表。')

prices = prices.sort_index()
prices_path = data_raw_dir / 'prices_raw.csv'
prices.to_csv(prices_path)

# 2) 获取标普 500 基准（扩展）
sp500 = yf.download(
    '^GSPC',
    period='5y',
    interval='1d',
    auto_adjust=True,
    progress=False
)['Close']
if sp500.empty:
    raise ValueError('标普 500 数据为空，请检查网络或数据源。')
sp500 = sp500.sort_index()
sp500_path = data_raw_dir / 'sp500_raw.csv'
sp500.to_csv(sp500_path)

# 3) 获取财务摘要
records = []
errors = []
for tk in tickers:
    try:
        info = yf.Ticker(tk).info
        records.append({
            'ticker': tk,
            'name': info.get('shortName'),
            'marketCap': info.get('marketCap'),
            'trailingPE': info.get('trailingPE'),
            'priceToBook': info.get('priceToBook'),
            'revenueGrowth': info.get('revenueGrowth'),
            'grossMargins': info.get('grossMargins'),
            'returnOnEquity': info.get('returnOnEquity'),
            'currency': info.get('currency'),
            'asOfDate': datetime.now(timezone.utc).date().isoformat()
        })
    except Exception as exc:
        errors.append((tk, str(exc)))

financials = pd.DataFrame(records)
financials_path = data_raw_dir / 'financials_raw.csv'
financials.to_csv(financials_path, index=False)

# 4) 数据质量与真实性检查
missing_ratio = prices.isna().mean().sort_values(ascending=False)
coverage = {
    'start_date': prices.index.min(),
    'end_date': prices.index.max(),
    'trading_days': len(prices.index)
}
duplicates = prices.index.duplicated().sum()
missing_columns = sorted(set(tickers) - set(prices.columns))
non_positive = (prices <= 0).sum().sort_values(ascending=False)
latest_gap_days = (datetime.now(timezone.utc).date() - prices.index.max().date()).days
outlier_check = (prices.pct_change().abs() > 0.2).sum().sort_values(ascending=False)

print('数据获取完成：', prices_path, financials_path, sp500_path)
print('\n缺失率（前 7 列）：')
print(missing_ratio.head(7))
print('\n时间覆盖：', coverage)
print('重复日期数：', duplicates)
print('非正价格计数（前 7 列）：')
print(non_positive.head(7))
print('|日收益率| > 20% 次数（前 7 列）：')
print(outlier_check.head(7))
print('最新交易日与今日间隔（天）：', latest_gap_days)
if missing_columns:
    print('缺失列：', missing_columns)
if errors:
    print('\n财务摘要获取错误：')
    for tk, msg in errors:
        print(f'- {tk}: {msg}')

C:\Users\HUAWEI\AppData\Local\Temp\ipykernel_33312\2594742166.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'asOfDate': datetime.utcnow().date().isoformat()


数据获取完成： D:\中大\第二学期\数字经济\T-A2_Magnificent7\data_raw\prices_raw.csv D:\中大\第二学期\数字经济\T-A2_Magnificent7\data_raw\financials_raw.csv

缺失率（前 7 列）：
Ticker
AAPL     0.0
AMZN     0.0
GOOGL    0.0
META     0.0
MSFT     0.0
NVDA     0.0
TSLA     0.0
dtype: float64

时间覆盖： {'start_date': Timestamp('2021-05-17 00:00:00'), 'end_date': Timestamp('2026-05-15 00:00:00'), 'trading_days': 1256}
重复日期数： 0


## 结果解读
- 数据是否完整覆盖近 5 年交易日？是否存在明显缺失？
- 缺失值主要集中在哪些股票？原因是什么（停牌、数据源字段缺失等）？
- 非正价格或异常大幅波动是否存在？若有，需在报告中说明并核实。
- 最新交易日与当前日期的间隔是否合理（节假日除外）？
- 财务摘要是否存在空值（如 PB、PE 为空），需要在报告中说明。